In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from data_preprocessing import find_split_run_by_name

from feature_engineering import FeatureEngineeringParams, engineer_and_save_windows

In [4]:
PREPROCESSED_ROOT = PROJECT_ROOT / 'dataset' / 'preprocessed'
OUTPUT_ROOT = PROJECT_ROOT / 'dataset' / 'processed_windows'

# Select the split by its folder name under dataset/preprocessed/splits.
SPLIT_RUN_NAME = '20260422_161847'

# Point mode: make each sample a "window" (equivalent to individual data points).
POINT_MODE = True
WINDOW_SIZE = 1 if POINT_MODE else 60
STRIDE = 1 if POINT_MODE else 10

# Window labeling config (used when POINT_MODE=False):
# - strategy='positive_ratio': use the whole window
# - strategy='last': use only the last WINDOW_LABEL_LAST_PERCENT of the window
WINDOW_LABEL_STRATEGY = 'last'
WINDOW_LABEL_POSITIVE_RATIO = 0.01
WINDOW_LABEL_LAST_PERCENT = 2

SPLIT_RUN_DIR = find_split_run_by_name(PREPROCESSED_ROOT, SPLIT_RUN_NAME)

TRAIN_SPLIT_PATH = SPLIT_RUN_DIR / 'train_split.csv'
VAL_SPLIT_PATH = SPLIT_RUN_DIR / 'val_split.csv'
TEST_SPLIT_PATH = SPLIT_RUN_DIR / 'test_split.csv'

params = FeatureEngineeringParams(
    train_csv_path=str(TRAIN_SPLIT_PATH),
    val_csv_path=str(VAL_SPLIT_PATH),
    test_csv_path=str(TEST_SPLIT_PATH),
    timestamp_col='timestamp',
    label_col='failure_label',
    train_normal_only=True,  # Enforced: resulting train labels must be all 0
    feature_cols=(),  # Empty => auto-select numeric features excluding label columns
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    flatten_windows=True,
    window_label_strategy=WINDOW_LABEL_STRATEGY,
    window_label_positive_ratio=WINDOW_LABEL_POSITIVE_RATIO,
    window_label_last_percent=WINDOW_LABEL_LAST_PERCENT,
    point_mode=POINT_MODE,
)
print(f'Using split directory: {SPLIT_RUN_DIR}')
params

Using split directory: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/preprocessed/splits/20260422_161847


FeatureEngineeringParams(train_csv_path='/Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/preprocessed/splits/20260422_161847/train_split.csv', val_csv_path='/Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/preprocessed/splits/20260422_161847/val_split.csv', test_csv_path='/Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/preprocessed/splits/20260422_161847/test_split.csv', timestamp_col='timestamp', label_col='failure_label', train_normal_only=True, feature_cols=(), window_size=1, stride=1, flatten_windows=True, window_label_strategy='last', window_label_positive_ratio=0.01, window_label_last_percent=2, point_mode=True)

In [5]:
artifacts = engineer_and_save_windows(params, OUTPUT_ROOT)

print(f'Run directory: {artifacts.run_dir}')
print(f'Train windows: {artifacts.train_windows_path}')
print(f'Val windows: {artifacts.val_windows_path}')
print(f'Test windows: {artifacts.test_windows_path}')
print(f'Train window labels: {artifacts.train_window_labels_path}')
print(f'Val window labels: {artifacts.val_window_labels_path}')
print(f'Test window labels: {artifacts.test_window_labels_path}')

metadata = json.loads(artifacts.metadata_path.read_text())
display(pd.json_normalize(metadata, sep='.').T.rename(columns={0: 'value'}))

Run directory: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223
Train windows: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/train_windows.npy
Val windows: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/val_windows.npy
Test windows: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/test_windows.npy
Train window labels: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/train_window_labels.npy
Val window labels: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/val_window_labels.npy
Test window labels: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223/test_window_labels.npy


,value
run_id,20260422_162223
feature_cols,"[TP2, TP3, H1, DV_pressure, Reservoirs, Oil_te..."
feature_count_warning,
train_rows,445298
val_rows,198734
test_rows,429314
train_candidate_windows,445298
train_kept_windows,445298
train_windows_shape,"[445298, 15]"
val_windows_shape,"[198734, 15]"


In [6]:
history_path = OUTPUT_ROOT / 'run_history.csv'
run_row = {
    'run_id': metadata['run_id'],
    'window_size': metadata['params']['window_size'],
    'stride': metadata['params']['stride'],
    'flatten_windows': metadata['params']['flatten_windows'],
    'train_rows': metadata['train_rows'],
    'val_rows': metadata['val_rows'],
    'test_rows': metadata['test_rows'],
    'train_windows_shape': str(metadata['train_windows_shape']),
    'val_windows_shape': str(metadata.get('val_windows_shape', '')),
    'test_windows_shape': str(metadata['test_windows_shape']),
    'train_window_labels': metadata['saved_files'].get('train_window_labels', ''),
    'val_window_labels': metadata['saved_files'].get('val_window_labels', ''),
    'test_window_labels': metadata['saved_files'].get('test_window_labels', ''),
}

new_row_df = pd.DataFrame([run_row])
if history_path.exists():
    old = pd.read_csv(history_path)
    history = pd.concat([old, new_row_df], ignore_index=True)
else:
    history = new_row_df

history.to_csv(history_path, index=False)
print(f'Updated history at: {history_path}')
display(history.tail(10))

Updated history at: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/run_history.csv


,run_id,window_size,stride,flatten_windows,train_rows,test_rows,train_windows_shape,test_windows_shape,train_window_labels,test_window_labels,val_rows,val_windows_shape,val_window_labels
4,20260421_211028,60,10,True,445298,411534,"[44524, 900]","[41148, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,0.0,"[0, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
5,20260421_212213,60,10,True,445298,212800,"[44524, 900]","[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,198734.0,"[19868, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
6,20260421_225006,60,10,True,445298,439152,"[44524, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,411534.0,"[41148, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
7,20260421_234700,60,10,True,644032,439152,"[63526, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
8,20260422_140303,60,10,True,644032,439152,"[63526, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
9,20260422_142527,60,10,True,644032,439152,"[63526, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
10,20260422_143854,60,10,True,644032,439152,"[63526, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
11,20260422_144602,60,10,True,644032,439152,"[63526, 900]","[43910, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[21275, 900]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
12,20260422_145745,1,1,True,644032,439152,"[635375, 15]","[439152, 15]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,212800.0,"[212800, 15]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
13,20260422_162223,1,1,True,445298,429314,"[445298, 15]","[429314, 15]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...,198734.0,"[198734, 15]",/Users/Leviathan/Downloads/DDA4210/vae-anomaly...


In [7]:
all_runs = sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir()])
print(f'Total saved runs: {len(all_runs)}')
for p in all_runs[-5:]:
    print(f' - {p.name}')

Total saved runs: 2
 - 20260422_144602
 - 20260422_162223


In [8]:
import numpy as np

def _label_counts_from_npy(path):
    labels = np.load(path)
    values, counts = np.unique(labels, return_counts=True)
    return {int(value): int(count) for value, count in zip(values, counts)}

for run_dir in all_runs:
    train_path = run_dir / 'train_window_labels.npy'
    val_path = run_dir / 'val_window_labels.npy'
    test_path = run_dir / 'test_window_labels.npy'

    print(f'Folder: {run_dir.name}')
    if train_path.exists():
        print(f"  train_window_labels: {_label_counts_from_npy(train_path)}")
    if val_path.exists():
        print(f"  val_window_labels: {_label_counts_from_npy(val_path)}")
    if test_path.exists():
        print(f"  test_window_labels: {_label_counts_from_npy(test_path)}")


Folder: 20260422_144602
  train_window_labels: {0: 63526}
  val_window_labels: {0: 21039, 1: 236}
  test_window_labels: {0: 42016, 1: 1894}
Folder: 20260422_162223
  train_window_labels: {0: 445298}
  val_window_labels: {0: 190077, 1: 8657}
  test_window_labels: {0: 409639, 1: 19675}
